# Section 15: Human-in-the-Loop in LangGraph

Is notebook mein hum seekhenge ki **LangGraph** ke andar **Human-in-the-Loop (HITL)** pattern kaise implement kiya jata hai:
- Breakpoints (`interrupt_before` / `interrupt_after`)
- State inspection using `get_state()`
- State modification using `update_state()`
- Paused graph ko resume karna (`stream(None, ...)`)


### 1. Environment Setup & LLM Initialization
- **Purpose**: `.env` file se API keys load karna aur Groq provider ke through Chat Model (`openai/gpt-oss-20b`) initialize karna.
- **Parameters**: `model='openai/gpt-oss-20b'`, `model_provider='groq'`.
- **Logic**: `dotenv.load_dotenv()` project root se environment variables load karta hai jisse API authentication complete ho sake.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY', '')

from langchain.chat_models import init_chat_model

# Initialize LLM with Groq provider
llm = init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')


### 2. Custom Math Tools Definition & Tool Binding
- **Purpose**: Agent ke liye basic mathematical operations (`multiply`, `addition`, `divide`) as tools create karna aur LLM ke saath bind karna.
- **Functions**:
  - `multiply(a: int, b: int) -> int`: Do integers ka product return karta hai.
  - `addition(a: int, b: int) -> int`: Do integers ka sum return karta hai.
  - `divide(a: int, b: int) -> float`: Do integers ka division return karta hai (with zero-division handling).
- **Logic**: `llm.bind_tools(tool)` se LLM model tool schemas ko recognize kar pata hai aur tool calls generate karta hai.


In [ ]:
# Custom Math Tools
def multiply(a: int, b: int) -> int:
    """Multiply a and b.
    Args:
        a: first int
        b: second int
    """
    return a * b

def addition(a: int, b: int) -> int:
    """Add a and b.
    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a by b.
    Args:
        a: first int
        b: second int
    """
    if b == 0:
        raise ValueError("Denominator cannot be zero.")
    return a / b

tool = [multiply, addition, divide]
llm_with_tool = llm.bind_tools(tool)


### 3. LangGraph Core Components Import
- **Purpose**: Graph creation, message handling, checkpointing aur tool prebuilts import karna.
- **Key Imports**:
  - `StateGraph`, `START`, `END`: Graph nodes aur flow control ke liye.
  - `ToolNode`, `tools_condition`: Automatic tool calling orchestration ke liye.
  - `MessagesState`: Standard chat messages state representation.
  - `MemorySaver`: Checkpointing persistence jiske bina breakpoints kaam nahi kar sakte.


In [ ]:
from langgraph.graph import START, StateGraph, END, MessagesState
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage


### 4. System Message Configuration
- **Purpose**: Assistant ke role aur behavioral guidelines define karna.
- **Logic**: Model ko strictly math operations perform karne aur politely answer karne ka prompt diya gaya hai.


In [ ]:
sys_msg = SystemMessage(content="Hey you are a math teacher. Perform only math operations, otherwise respond humbly.")


### 5. Assistant Node Function
- **Purpose**: Current conversation messages ke saath system prompt prepend karke LLM call karna.
- **Parameters**: `state: MessagesState` - Graph ki current state jisme message history hoti hai.
- **Returns**: `dict` - `{'messages': [response]}` jo existing messages list mein append ho jata hai.


In [ ]:
def assistant(state: MessagesState):
    """Assistant node that prepends the system prompt and invokes the tool-bound LLM."""
    return {'messages': [llm_with_tool.invoke([sys_msg] + state['messages'])]}


### 6. MemorySaver Checkpointer Initialization
- **Purpose**: In-memory checkpointer initialize karna jo state snapshots persist karta hai.
- **Important Logic**: Human-in-the-loop mein execution pause aur resume karne ke liye checkpointer mandatory hota hai.


In [ ]:
memory = MemorySaver()


### 7. StateGraph Construction & Routing
- **Purpose**: Nodes add karna aur unke beech explicit directed edges define karna.
- **Flow Logic**:
  - `START -> assistant`: Execution assistant node se shuru hoti hai.
  - `assistant -> tools_condition`: `tools_condition` check karta hai agar tool calls hain toh `tools` node par redirect kare, warna `END` par.
  - `tools -> assistant`: Tools execute hone ke baad output wapas assistant ke paas jaata hai final reply generate karne ke liye.
> **Error Fix**: Pehle code mein redundant `graph.add_edge('assistant', END)` aur `graph.add_edge('tools', END)` likha hua tha jisse parallel conflicting transitions ho rahe the aur tool run hone ke baad assistant ka final response skip ho raha tha. Ab ye clean aur standard pattern par configured hai.


In [ ]:
# Build the StateGraph
graph = StateGraph(MessagesState)

# Add nodes
graph.add_node('assistant', assistant)
graph.add_node('tools', ToolNode(tool))

# Add edges
graph.add_edge(START, 'assistant')
graph.add_conditional_edges('assistant', tools_condition)
graph.add_edge('tools', 'assistant')


### 8. Graph Compilation with Breakpoint (`interrupt_before`)
- **Purpose**: Graph ko compile karna aur `interrupt_before=['assistant']` set karke human intervention breakpoint lagana.
- **Breakpoint Logic**: Har baar jab graph `assistant` node execute karne wala hoga, execution pause ho jayegi.


In [ ]:
# Compile graph with breakpoint before assistant
graph = graph.compile(interrupt_before=['assistant'], checkpointer=memory)


### 9. Thread Configuration Setup
- **Purpose**: Execution context aur checkpointer identification ke liye session `thread_id` define karna.


In [ ]:
thread_id = {'configurable': {'thread_id': '123'}}


### 10. Initial Invocation (Triggering the Breakpoint)
- **Purpose**: Graph ko initial query provide karna (`tell me 2+3`).
- **Behavior**: Kyunki `interrupt_before=['assistant']` hai, graph start node se input accept karega aur assistant run hone se theek pehle pause ho jayega.


In [ ]:
initial_query = {'messages': [HumanMessage(content='tell me 2+3')]}

for event in graph.stream(initial_query, config=thread_id, stream_mode='values'):
    event['messages'][-1].pretty_print()


### 11. State Inspection: Pending Next Node
- **Purpose**: Check karna ki graph paused state mein hai aur next execution node kaunsa hai.
- **Expected Value**: `('assistant',)` - Shows ki assistant execute hone ke liye queue mein hai.


In [ ]:
state = graph.get_state(thread_id)
print('Next node to execute:', state.next)


### 12. Checkpoint History Inspection
- **Purpose**: Thread ke previous checkpoint snapshots aur state metadata check karna.


In [ ]:
state_history = list(graph.get_state_history(thread_id))
print(f'Total checkpoints saved: {len(state_history)}')
state_history[0]


### 13. Resuming Execution (Human Approval)
- **Purpose**: Paused graph ko resume karna by passing `None` as input.
- **Logic**: Assistant run hoga, addition tool call karega, aur tool result aane ke baad wapas assistant node se pehle pause hoga.


In [ ]:
# Resume graph execution (Human approved)
for event in graph.stream(None, config=thread_id, stream_mode='values'):
    event['messages'][-1].pretty_print()


### 14. Completing Final Response (After Tool Execution)
- **Purpose**: Tool message aane ke baad assistant ko resume karna taaki wo user ko final answer deliver kar sake.


In [ ]:
# Check if next node is pending and resume
print('Pending node:', graph.get_state(thread_id).next)
for event in graph.stream(None, config=thread_id, stream_mode='values'):
    event['messages'][-1].pretty_print()


## Editing State: Human Feedback (`update_state`)
Ab hum dekhenge ki graph pause hone par **Human Feedback** ke zariye state ko kaise change/override kiya jata hai.


### 15. Initial Query for Thread '1'
- **Purpose**: Thread '1' mein request bhejna: `'multiply 2 and 3'`.
- **Behavior**: Kyunki breakpoint active hai, graph assistant se pehle ruk jayega.


In [ ]:
initial_input = {'messages': [HumanMessage(content='multiply 2 and 3')]}
thread = {'configurable': {'thread_id': '1'}}

for event in graph.stream(initial_input, thread, stream_mode='values'):
    event['messages'][-1].pretty_print()


### 16. Verify Paused State
- **Purpose**: Confirm karna ki thread '1' par graph `assistant` par paused hai.


In [ ]:
state = graph.get_state(thread)
print('Pending next node:', state.next)


### 17. Updating State with Human Feedback
- **Purpose**: User ne apna mind change kiya aur instruction update kar di: `'NO please multiply 15 and 6'`.
- **Method**: `graph.update_state(thread, {'messages': [HumanMessage(content='...')]})`


In [ ]:
# Human modifies state before assistant runs
graph.update_state(thread, {'messages': [HumanMessage(content='NO please multiply 15 and 6')]})


### 18. Verifying Updated State Messages
- **Purpose**: State check karna aur verify karna ki naya HumanMessage conversation history mein append ho chuka hai.


In [ ]:
new_state = graph.get_state(thread).values
for msg in new_state['messages']:
    msg.pretty_print()


### 19. Resuming Execution with Updated State
- **Purpose**: Paused thread ko resume karna taaki assistant updated request (`15 * 6`) ko process kare aur tool call kare.


In [ ]:
for event in graph.stream(None, thread, stream_mode='values'):
    event['messages'][-1].pretty_print()


### 20. Resuming to Final AI Output
- **Purpose**: Tool message aane ke baad assistant ko complete hone dena taaki wo final calculation answer deliver kare.


In [ ]:
# Final resume to let assistant output the answer
if graph.get_state(thread).next:
    for event in graph.stream(None, thread, stream_mode='values'):
        event['messages'][-1].pretty_print()
else:
    print('Workflow completed!')
